# Word Alignment — Pairwise Levenshtein Normalisation

Determines the **best word alignment** for each source word against the DAT and DIT transcripts & uses pairwise Levenshtein normalization (cf. cross-comparison.ipynb).

Scoring methods:
- **Combined Weighted Score** — $\alpha \cdot \text{lev\_sim} + (1-\alpha) \cdot \text{pos\_sim}$, $\alpha = 0.7$
- **Combined Harmonic Score** — $\frac{2 \cdot \text{lev\_sim} \cdot \text{pos\_sim}}{\text{lev\_sim} + \text{pos\_sim}}$

## Setup

In [45]:
import sys
import importlib
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("__file__").resolve().parent))

import utils
importlib.reload(utils)

from utils import build_word_comparison_df, clean

In [46]:
PROJECT_ROOT = Path("__file__").resolve().parent.parent
ANALYSIS_DIR = PROJECT_ROOT / "02-analysis-position"
DIT_TSV = ANALYSIS_DIR / "dialect-ignorant-transcript.tsv"
DAT_TSV = ANALYSIS_DIR / "dialect-aware-transcript.tsv"

## Load Data

In [47]:
df_dit = pd.read_csv(DIT_TSV, sep='\t', encoding='utf-8-sig')
df_dat = pd.read_csv(DAT_TSV, sep='\t', encoding='utf-8-sig')
df = pd.merge(df_dit, df_dat[['path', 'DAT']], on='path', how='inner')

print(f"Clips: {len(df)}")
df.head()

Clips: 100


,path,duration,sentence,sentence_source,client_id,dialect_region,canton,zipcode,age,gender,DIT,DAT
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,4.778662,Insgesamt habe ich einige Hunderttausend Frank...,news,300bb931-79ae-40ec-b989-3efd5e83f4c2,Zürich,ZH,8408.0,fourties,male,Insgesamt habe ich einige hundert Dusik vor un...,Insgesamt habe ich einige 100.000 Franken verl...
1,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dca...,5.294150,Welche Rolle hatte Rosenberg während des Holoc...,news,6e084270-8d26-43d9-ba69-5e8ee793ab8c,Zürich,ZG,6340.0,twenties,female,"Weli Rolle, höchter Uz stoodhergewerden von Ho...",Wer ironne hättet viele bands auf den einzugem...
2,clips\6858a37b-edd0-4fdf-871c-96d3b1bd3e21\82f...,5.802676,Das ist angesichts aller schlechten Optionen d...,news,6858a37b-edd0-4fdf-871c-96d3b1bd3e21,Zürich,ZH,8704.0,fourties,male,Das ist angesichts vor allen Schlachten Option...,Das ist angesichts von allen schlechten Option...
3,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\a21...,5.973333,Ebenfalls keinen Sieg durfte die AC Milan feiern.,news,c4c03e6f-50a2-4d24-ae88-caf3032798fa,Zürich,AG,4663.0,fourties,male,Aber voll ScreenZig hat 100 000 derumphierig,Aber falls kein Sieg hat AC Milan Dörfe 4.
4,clips\f877ee50-af2b-423b-bdda-0cc406032c45\4e5...,4.778662,Viele Fussballfans haben die Alte Reithalle un...,parliament,f877ee50-af2b-423b-bdda-0cc406032c45,Zürich,ZH,8405.0,fourties,female,Viele Fußballfans hängt die Alldri-Taller unte...,Viele Fußballfans handt die All 3 Taller unter...


## Cross-Comparison with Pairwise NED

In [48]:
# max_word_len=None → pairwise normalisation per word pair (standard NED)
word_df = build_word_comparison_df(df, max_word_len=None)
print(f"Total rows: {len(word_df):,}")
word_df.head(10)

Total rows: 9,441


,clip_id,src_word_index,target_word_index,is_same_position,position_score,src_word,dit_word,dat_word,dit_lev_sim,dat_lev_sim,dit_combined_weighted,dit_combined_harmonic,dat_combined_weighted,dat_combined_harmonic,dat_advantage,src_len,dit_len,dat_len
0,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,0,1,1.000,insgesamt,insgesamt,insgesamt,1.000,1.000,1.000,1.000,1.000,1.000,0.000,7,9,7
1,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,1,0,0.875,insgesamt,habe,habe,0.111,0.111,0.340,0.197,0.340,0.197,0.000,7,9,7
2,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,2,0,0.750,insgesamt,ich,ich,0.111,0.111,0.303,0.193,0.303,0.193,0.000,7,9,7
3,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,3,0,0.625,insgesamt,einige,einige,0.333,0.333,0.421,0.434,0.421,0.434,0.000,7,9,7
4,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,4,0,0.500,insgesamt,hundert,100000,0.222,0.000,0.305,0.307,0.150,0.000,-0.222,7,9,7
5,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,5,0,0.375,insgesamt,dusik,franken,0.111,0.000,0.190,0.171,0.113,0.000,-0.111,7,9,7
6,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,6,0,0.250,insgesamt,vor,verloren,0.000,0.000,0.075,0.000,0.075,0.000,0.000,7,9,7
7,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,7,0,0.125,insgesamt,und,None,0.111,0.000,0.115,0.118,0.038,0.000,-0.111,7,9,7
8,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,0,8,0,0.000,insgesamt,verloren,None,0.000,0.000,0.000,0.000,0.000,0.000,0.000,7,9,7
9,clips\300bb931-79ae-40ec-b989-3efd5e83f4c2\83c...,1,0,0,0.875,habe,insgesamt,insgesamt,0.111,0.111,0.340,0.197,0.340,0.197,0.000,7,9,7


## Best Word Alignment: Combined Weighted Score

For each source word, the row with the **highest `dat_combined_weighted` or `dit_combined_weighted`** is selected as the best alignment.

In [49]:
dat_best_w = (
    word_df.loc[word_df.groupby(['clip_id', 'src_word_index'])['dat_combined_weighted'].idxmax()]
    [['clip_id', 'src_word_index', 'src_word', 'dat_word',
      'dat_lev_sim', 'position_score', 'dat_combined_weighted']]
    .rename(columns={'dat_word': 'aligned_word', 'dat_lev_sim': 'lev_sim',
                     'dat_combined_weighted': 'score'})
    .assign(model='DAT')
)

dit_best_w = (
    word_df.loc[word_df.groupby(['clip_id', 'src_word_index'])['dit_combined_weighted'].idxmax()]
    [['clip_id', 'src_word_index', 'src_word', 'dit_word',
      'dit_lev_sim', 'position_score', 'dit_combined_weighted']]
    .rename(columns={'dit_word': 'aligned_word', 'dit_lev_sim': 'lev_sim',
                     'dit_combined_weighted': 'score'})
    .assign(model='DIT')
)

best_weighted = pd.concat([dit_best_w, dat_best_w], ignore_index=True)

print(f"Total alignments (weighted): {len(best_weighted):,}")
print("\nTop 20 — highest Combined Weighted Score:")
best_weighted.sort_values('score', ascending=False).head(20)[
    ['clip_id', 'src_word', 'aligned_word', 'lev_sim', 'position_score', 'score', 'model']
]

Total alignments (weighted): 1,682

Top 20 — highest Combined Weighted Score:


,clip_id,src_word,aligned_word,lev_sim,position_score,score,model
48,clips\0d7bacea-1a75-4c14-b580-40d5b98663b6\d6d...,muss,muss,1.0,1.0,1.0,DIT
1658,clips\f975c67c-35a5-4cac-8195-51f456b59edc\3be...,später,später,1.0,1.0,1.0,DAT
7,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\473...,aber,aber,1.0,1.0,1.0,DIT
1681,clips\f975c67c-35a5-4cac-8195-51f456b59edc\422...,haben,haben,1.0,1.0,1.0,DAT
9,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\473...,ist,ist,1.0,1.0,1.0,DIT
23,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\aed...,ich,ich,1.0,1.0,1.0,DIT
1678,clips\f975c67c-35a5-4cac-8195-51f456b59edc\422...,eine,eine,1.0,1.0,1.0,DAT
1679,clips\f975c67c-35a5-4cac-8195-51f456b59edc\422...,schöne,schöne,1.0,1.0,1.0,DAT
1680,clips\f975c67c-35a5-4cac-8195-51f456b59edc\422...,zukunft,zukunft,1.0,1.0,1.0,DAT
24,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\aed...,habe,habe,1.0,1.0,1.0,DIT


In [50]:
cols = ['clip_id', 'src_word', 'aligned_word', 'lev_sim', 'position_score', 'score', 'model']
sorted_w = best_weighted.sort_values('score', ascending=False)

near_perfect_w = best_weighted[best_weighted['lev_sim'] < 1.0].sort_values('lev_sim', ascending=False).head(10)
print("Top 10 — lev_sim closest to 1 but not exactly 1 (weighted):")
display(near_perfect_w[cols].reset_index(drop=True))

print("\nBottom 10 — worst Combined Weighted Score:")
display(sorted_w.tail(10)[cols].reset_index(drop=True))

Top 10 — lev_sim closest to 1 but not exactly 1 (weighted):


,clip_id,src_word,aligned_word,lev_sim,position_score,score,model
0,clips\2346f20d-b2bc-4712-8c83-5e376731a1c9\64e...,freiheitsstrafe,freiheitstrafe,0.933,0.889,0.920,DIT
1,clips\2346f20d-b2bc-4712-8c83-5e376731a1c9\64e...,freiheitsstrafe,freiheitstrafe,0.933,1.000,0.953,DAT
2,clips\f975c67c-35a5-4cac-8195-51f456b59edc\06c...,notwendigste,notwendigsten,0.923,0.969,0.937,DAT
3,clips\3ce6b6dc-6c03-48c1-83bd-024bbecfaf9a\d38...,entsprechend,entsprechen,0.917,1.000,0.942,DAT
4,clips\e6a8968c-d53c-4f76-80b5-15923136987b\1ce...,allermeisten,allermeiste,0.917,1.000,0.942,DIT
5,clips\7c882fa4-86f6-4986-b211-295a09265deb\c3f...,produzenten,produzentenz,0.917,0.857,0.899,DAT
6,clips\c81c27c4-3e50-4b38-827e-2fc2bf899dc7\723...,niederlagen,niederlage,0.909,0.800,0.876,DAT
7,clips\c81c27c4-3e50-4b38-827e-2fc2bf899dc7\723...,niederlagen,niederlage,0.909,0.900,0.906,DIT
8,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\b47...,unternehmen,unternehmer,0.909,1.000,0.936,DIT
9,clips\e6a8968c-d53c-4f76-80b5-15923136987b\1ce...,kandidaten,kandidatin,0.900,0.800,0.870,DIT



Bottom 10 — worst Combined Weighted Score:


,clip_id,src_word,aligned_word,lev_sim,position_score,score,model
0,clips\0f19ca1d-11fe-4822-83e9-4d0f7bcce0cd\954...,strafanzeige,sind,0.167,0.500,0.267,DAT
1,clips\e4d6b98b-b3f3-4e0f-9954-4abb7de23540\48b...,tatsächlichen,wright,0.154,0.529,0.267,DAT
2,clips\f877ee50-af2b-423b-bdda-0cc406032c45\5be...,zu,dürz,0.000,0.889,0.267,DAT
3,clips\e4d6b98b-b3f3-4e0f-9954-4abb7de23540\48b...,informieren,wright,0.182,0.412,0.251,DAT
4,clips\0f19ca1d-11fe-4822-83e9-4d0f7bcce0cd\954...,strafanzeige,None,0.000,0.833,0.250,DIT
5,clips\68931ace-d879-48a2-b92b-0630ff454bdf\d34...,leben,None,0.000,0.800,0.240,DIT
6,clips\865f69a7-acb2-4604-a356-cdad6517606c\f35...,kurios,savor,0.167,0.286,0.203,DAT
7,clips\0f19ca1d-11fe-4822-83e9-4d0f7bcce0cd\954...,eingereicht,None,0.000,0.667,0.200,DIT
8,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\469...,erinnern,None,0.000,0.667,0.200,DAT
9,clips\865f69a7-acb2-4604-a356-cdad6517606c\f35...,sei,benötig,0.286,0.000,0.200,DAT


## Best Word Alignment: Combined Harmonic Score

For each source word, the row with the **highest `dat_combined_harmonic` or `dit_combined_harmonic`** is selected as the best alignment.

In [51]:
dat_best_h = (
    word_df.loc[word_df.groupby(['clip_id', 'src_word_index'])['dat_combined_harmonic'].idxmax()]
    [['clip_id', 'src_word_index', 'src_word', 'dat_word',
      'dat_lev_sim', 'position_score', 'dat_combined_harmonic']]
    .rename(columns={'dat_word': 'aligned_word', 'dat_lev_sim': 'lev_sim',
                     'dat_combined_harmonic': 'score'})
    .assign(model='DAT')
)

dit_best_h = (
    word_df.loc[word_df.groupby(['clip_id', 'src_word_index'])['dit_combined_harmonic'].idxmax()]
    [['clip_id', 'src_word_index', 'src_word', 'dit_word',
      'dit_lev_sim', 'position_score', 'dit_combined_harmonic']]
    .rename(columns={'dit_word': 'aligned_word', 'dit_lev_sim': 'lev_sim',
                     'dit_combined_harmonic': 'score'})
    .assign(model='DIT')
)

best_harmonic = pd.concat([dit_best_h, dat_best_h], ignore_index=True)

print(f"Total alignments (harmonic): {len(best_harmonic):,}")
print("\nTop 20 — highest Combined Harmonic Score:")
best_harmonic.sort_values('score', ascending=False).head(20)[
    ['clip_id', 'src_word', 'aligned_word', 'lev_sim', 'position_score', 'score', 'model']
]

Total alignments (harmonic): 1,682

Top 20 — highest Combined Harmonic Score:


,clip_id,src_word,aligned_word,lev_sim,position_score,score,model
1658,clips\f975c67c-35a5-4cac-8195-51f456b59edc\3be...,später,später,1.0,1.0,1.0,DAT
7,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\473...,aber,aber,1.0,1.0,1.0,DIT
1678,clips\f975c67c-35a5-4cac-8195-51f456b59edc\422...,eine,eine,1.0,1.0,1.0,DAT
1679,clips\f975c67c-35a5-4cac-8195-51f456b59edc\422...,schöne,schöne,1.0,1.0,1.0,DAT
1680,clips\f975c67c-35a5-4cac-8195-51f456b59edc\422...,zukunft,zukunft,1.0,1.0,1.0,DAT
1217,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dd9...,sich,sich,1.0,1.0,1.0,DAT
1218,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dd9...,das,das,1.0,1.0,1.0,DAT
1219,clips\6e084270-8d26-43d9-ba69-5e8ee793ab8c\dd9...,leisten,leisten,1.0,1.0,1.0,DAT
1666,clips\f975c67c-35a5-4cac-8195-51f456b59edc\3d0...,das,das,1.0,1.0,1.0,DAT
10,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\473...,doch,doch,1.0,1.0,1.0,DIT


In [52]:
sorted_h = best_harmonic.sort_values('score', ascending=False)

near_perfect_h = best_harmonic[best_harmonic['lev_sim'] < 1.0].sort_values('lev_sim', ascending=False).head(10)
print("Top 10 — lev_sim closest to 1 but not exactly 1 (harmonic):")
display(near_perfect_h[cols].reset_index(drop=True))

print("\nBottom 10 — worst Combined Harmonic Score:")
display(sorted_h.tail(10)[cols].reset_index(drop=True))

Top 10 — lev_sim closest to 1 but not exactly 1 (harmonic):


,clip_id,src_word,aligned_word,lev_sim,position_score,score,model
0,clips\2346f20d-b2bc-4712-8c83-5e376731a1c9\64e...,freiheitsstrafe,freiheitstrafe,0.933,1.000,0.965,DAT
1,clips\2346f20d-b2bc-4712-8c83-5e376731a1c9\64e...,freiheitsstrafe,freiheitstrafe,0.933,0.889,0.910,DIT
2,clips\f975c67c-35a5-4cac-8195-51f456b59edc\06c...,notwendigste,notwendigsten,0.923,0.969,0.945,DAT
3,clips\e6a8968c-d53c-4f76-80b5-15923136987b\1ce...,allermeisten,allermeiste,0.917,1.000,0.957,DIT
4,clips\7c882fa4-86f6-4986-b211-295a09265deb\c3f...,produzenten,produzentenz,0.917,0.857,0.886,DAT
5,clips\3ce6b6dc-6c03-48c1-83bd-024bbecfaf9a\d38...,entsprechend,entsprechen,0.917,1.000,0.957,DAT
6,clips\c81c27c4-3e50-4b38-827e-2fc2bf899dc7\723...,niederlagen,niederlage,0.909,0.900,0.904,DIT
7,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\b47...,unternehmen,unternehmer,0.909,1.000,0.952,DIT
8,clips\c81c27c4-3e50-4b38-827e-2fc2bf899dc7\723...,niederlagen,niederlage,0.909,0.800,0.851,DAT
9,clips\6858a37b-edd0-4fdf-871c-96d3b1bd3e21\82f...,schlechten,schlachten,0.900,0.875,0.887,DIT



Bottom 10 — worst Combined Harmonic Score:


,clip_id,src_word,aligned_word,lev_sim,position_score,score,model
0,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\705...,zu,der,0.000,0.250,0.0,DAT
1,clips\460a10ef-020f-4559-810a-115a066b9bab\a87...,,es,0.000,0.357,0.0,DIT
2,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\ad9...,hören,guijungs,0.125,0.000,0.0,DIT
3,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\ad9...,nacht,guijungs,0.000,0.667,0.0,DIT
4,clips\c81c27c4-3e50-4b38-827e-2fc2bf899dc7\cd3...,ersten,1,0.000,0.857,0.0,DIT
5,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\ad9...,der,guijungs,0.000,0.833,0.0,DIT
6,clips\00fafc97-bf8b-4f35-ae97-b22e83e150f2\705...,zu,da,0.000,0.250,0.0,DIT
7,clips\2be89650-fc8e-429a-a8e3-2e92b3538639\d72...,13,das,0.000,0.727,0.0,DIT
8,clips\c4c03e6f-50a2-4d24-ae88-caf3032798fa\ad9...,zu,in,0.000,0.167,0.0,DAT
9,clips\c81c27c4-3e50-4b38-827e-2fc2bf899dc7\cd3...,echt,1,0.000,0.286,0.0,DIT
